<a href="https://colab.research.google.com/github/SunnyChoudhary850/FlyRank/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
%pip -q install duckdb huggingface_hub

In [11]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

In [12]:
con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']}").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


**Signal 1 verdict: MIXED**

Decline rate does NOT increase steadily with staleness as the refresh-flag
logic assumes. Under-30-day pages actually have the HIGHEST decline rate
(0.689), the 30-90 day bucket has the LOWEST (0.541), and over-90-days sits
in between (0.647). This is not a clean "staler = worse" pattern — recency
of update alone doesn't cleanly predict decline in this data. A refresh
rule based purely on staleness would misfire often.

In [13]:
staleness_check = con.sql(f"""
    WITH content_with_bucket AS (
        SELECT content_hash_id,
               content_updated_date,
               CURRENT_DATE - content_updated_date AS days_since_update,
               CASE
                   WHEN CURRENT_DATE - content_updated_date < 30 THEN '1_under_30d'
                   WHEN CURRENT_DATE - content_updated_date < 90 THEN '2_30_90d'
                   ELSE '3_over_90d'
               END AS staleness_bucket
        FROM {TABLES['dim_content']}
    ),
    march_perf AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS impressions_march
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-03' AND gsc_data_available IS TRUE
        GROUP BY content_hash_id
    ),
    april_perf AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS impressions_april
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-04' AND gsc_data_available IS TRUE
        GROUP BY content_hash_id
    )
    SELECT c.staleness_bucket,
           COUNT(*) AS n,
           AVG(CASE WHEN a.impressions_april < m.impressions_march THEN 1 ELSE 0 END) AS decline_rate
    FROM content_with_bucket c
    JOIN march_perf m ON c.content_hash_id = m.content_hash_id
    JOIN april_perf a ON c.content_hash_id = a.content_hash_id
    GROUP BY c.staleness_bucket
    ORDER BY c.staleness_bucket
""").df()
staleness_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,staleness_bucket,n,decline_rate
0,1_under_30d,31408,0.688837
1,2_30_90d,95092,0.540718
2,3_over_90d,32049,0.646697


In [14]:
volume_check = con.sql(f"""
    WITH content_with_bucket AS (
        SELECT content_hash_id,
               search_volume,
               CASE
                   WHEN search_volume < 100 THEN '1_low'
                   WHEN search_volume < 1000 THEN '2_medium'
                   ELSE '3_high'
               END AS volume_bucket
        FROM {TABLES['dim_content']}
    ),
    march_perf AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_march
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-03' AND gsc_data_available IS TRUE
        GROUP BY content_hash_id
    ),
    april_perf AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_april
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-04' AND gsc_data_available IS TRUE
        GROUP BY content_hash_id
    )
    SELECT c.volume_bucket,
           COUNT(*) AS n,
           AVG(CASE WHEN a.impressions_april < m.impressions_march THEN 1 ELSE 0 END) AS decline_rate
    FROM content_with_bucket c
    JOIN march_perf m ON c.content_hash_id = m.content_hash_id
    JOIN april_perf a ON c.content_hash_id = a.content_hash_id
    GROUP BY c.volume_bucket
    ORDER BY c.volume_bucket
""").df()
volume_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,volume_bucket,n,decline_rate
0,1_low,133548,0.589541
1,2_medium,12822,0.596631
2,3_high,12179,0.607357


**Signal 2 verdict: FALSE**

Decline rate is nearly flat across volume buckets (0.590 low, 0.597 medium,
0.607 high, n=133548/12822/12179). Volume alone shows no meaningful
separation between declining and stable pages — a "quick win" rule based on
volume alone would not actually identify pages that need attention.

In [15]:
scoring = con.sql(f"""
    SELECT
        content_hash_id,
        content_updated_date,
        search_volume,
        CURRENT_DATE - content_updated_date AS days_since_update
    FROM {TABLES['dim_content']}
""").df()

# Score: purely based on staleness (the flag-linked signal), even though
# it came back MIXED -- being honest that this is the best available
# flag-linked signal, not a clean one. Normalize days_since_update to a
# 0-1 score, capped at 365 days.
scoring['score'] = (scoring['days_since_update'].clip(upper=365) / 365).round(3)

scoring['reason_code'] = 'STALE_CONTENT'
scoring['action'] = scoring['days_since_update'].apply(
    lambda d: 'REFRESH_CANDIDATE' if d > 90 else 'MONITOR'
)

ranked_queue = scoring.sort_values('score', ascending=False).reset_index(drop=True)

import os
os.makedirs('work/outputs', exist_ok=True)
ranked_queue.to_csv('work/outputs/baseline_action_score.csv', index=False)

ranked_queue.head(10)

,content_hash_id,content_updated_date,search_volume,days_since_update,score,reason_code,action
0,content_7627af027cb71971,2024-12-17,<NA>,588,1.0,STALE_CONTENT,REFRESH_CANDIDATE
1,content_7628e8cb365e9552,2024-11-13,<NA>,622,1.0,STALE_CONTENT,REFRESH_CANDIDATE
2,content_762b5d8227aa46a0,2024-12-13,<NA>,592,1.0,STALE_CONTENT,REFRESH_CANDIDATE
3,content_762c420f2369579d,2024-11-23,<NA>,612,1.0,STALE_CONTENT,REFRESH_CANDIDATE
4,content_762de44d4118d050,2024-11-11,<NA>,624,1.0,STALE_CONTENT,REFRESH_CANDIDATE
5,content_762f421f2387cb12,2024-11-26,<NA>,609,1.0,STALE_CONTENT,REFRESH_CANDIDATE
6,content_762f6ca3ce4fadba,2024-11-14,<NA>,621,1.0,STALE_CONTENT,REFRESH_CANDIDATE
7,content_bfc6977c32698e47,2024-11-23,<NA>,612,1.0,STALE_CONTENT,REFRESH_CANDIDATE
8,content_bfc62c65a89403ea,2024-11-30,<NA>,605,1.0,STALE_CONTENT,REFRESH_CANDIDATE
9,content_bfc5de5a48691dc5,2024-11-28,<NA>,607,1.0,STALE_CONTENT,REFRESH_CANDIDATE


## Top-10 Review

1. content_7627af027cb71971 — REFRESH_CANDIDATE — 588 days since update,
   score capped at 1.0. Would be wrong if this page has no real search
   demand — search_volume is missing (NA) for this row, so I can't confirm
   anyone is even looking for it.
2. content_7628e8cb365e9552 — REFRESH_CANDIDATE — 622 days since update.
   Same missing search_volume issue — refreshing a page nobody searches
   for wastes review time regardless of how stale it is.
3. content_762b5d8227aa46a0 — REFRESH_CANDIDATE — 592 days since update.
   Would be wrong if this is intentionally evergreen/reference content that
   isn't meant to change often.
4. content_762c420f2369579d — REFRESH_CANDIDATE — 612 days since update.
   Given my MIXED verdict on staleness overall, this page could easily
   still be performing fine despite its age — staleness alone doesn't
   confirm decline.
5. content_762de44d4118d050 — REFRESH_CANDIDATE — 624 days since update,
   the single oldest page in this list. Still missing search_volume, so
   even being "most stale" doesn't mean "most worth fixing."
6. content_762f421f2387cb12 — REFRESH_CANDIDATE — 609 days since update.
   Would be wrong if this page's content_updated_date is simply inaccurate
   or wasn't updated by a real content change (e.g. a metadata-only edit).
7. content_762f6ca3ce4fadba — REFRESH_CANDIDATE — 621 days since update.
   Same reasoning as #6 — an unreliable update timestamp would make this
   whole rule misfire.
8. content_bfc6977c32698e47 — REFRESH_CANDIDATE — 612 days since update,
   different content prefix than most others in this list (bfc vs 762) —
   worth checking if this is a different content type where staleness
   means something different.
9. content_bfc62c65a89403ea — REFRESH_CANDIDATE — 605 days since update.
   Would be wrong for the same reason as #8 — the rule doesn't account for
   content_type differences.
10. content_bfc5de5a48691dc5 — REFRESH_CANDIDATE — 607 days since update.
    Same missing search_volume problem — this entire top-10 list is really
    an arbitrary slice of a much larger tied group, since every score here
    is capped at exactly 1.0.

**Honest limitation of this rule:** all 10 rows scored a tied 1.0 because
`days_since_update` exceeded my 365-day cap — meaning this "top 10" isn't
truly ranked, it's an arbitrary sample from hundreds of equally-scored
rows. Also, missing `search_volume` for every single one of these rows
means I genuinely can't confirm any of them matter to a real searcher.

## Self-Check

- [x] Two signal verdicts given with visible bucket tables and n (staleness:
  MIXED, n=31408/95092/32049; volume: FALSE, n=133548/12822/12179)
- [x] At least one signal is flag-linked (staleness → refresh flags)
- [x] One rule encoded: score, reason code (STALE_CONTENT), action label
  (REFRESH_CANDIDATE / MONITOR)
- [x] Ranked queue written to work/outputs/baseline_action_score.csv
- [x] Ten reviewed rows, each with action + why + what would make it wrong
- [x] No future-window or label-derived inputs used in the rule itself